In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable
import re

storage_account = "sanassign"

bronze_base = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/sales-view"
)

silver_base = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/sales-view"
)

gold_base = (
    f"abfss://gold@{storage_account}.dfs.core.windows.net/sales-view"
)

def to_snake_case(name):
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_").lower()

def snake_case_columns(df):
    return df.toDF(*[to_snake_case(c) for c in df.columns])

def read_csv(path):
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .csv(path)
    )

def normalize_date(df, column_name):
    if column_name not in df.columns:
        return df

    parsed_timestamp = F.coalesce(
        F.to_timestamp(F.col(column_name), "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(F.col(column_name), "yyyy-MM-dd HH:mm"),
        F.to_timestamp(F.col(column_name), "yyyy-MM-dd"),
        F.to_timestamp(F.col(column_name), "dd-MM-yyyy HH:mm:ss"),
        F.to_timestamp(F.col(column_name), "dd-MM-yyyy")
    )

    return df.withColumn(
        column_name,
        F.date_format(parsed_timestamp, "yyyy-MM-dd")
    )

def upsert_delta(df, target_path, merge_condition):
    if DeltaTable.isDeltaTable(spark, target_path):
        target = DeltaTable.forPath(spark, target_path)

        (
            target.alias("target")
            .merge(df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(target_path)
        )

In [0]:
product = read_csv(f"{bronze_base}/product")
product = snake_case_columns(product)

product.printSchema()
display(product)

In [0]:
product = product.withColumn(
    "sub_category",
    F.when(F.col("category_id").cast("int") == 1, "phone")
     .when(F.col("category_id").cast("int") == 2, "laptop")
     .when(F.col("category_id").cast("int") == 3, "playstation")
     .when(F.col("category_id").cast("int") == 4, "e-device")
     .otherwise("unknown")
)

In [0]:
product = normalize_date(product, "product_created_at")
product = normalize_date(product, "product_updated_at")
product = normalize_date(product, "expiry_date")

In [0]:
product_path = f"{silver_base}/product"

upsert_delta(
    product,
    product_path,
    "target.product_id = source.product_id"
)

In [0]:
display(product)

In [0]:
store = read_csv(f"{bronze_base}/store")
store = snake_case_columns(store)

store.printSchema()
display(store)

In [0]:
store = store.withColumn(
    "store_category",
    F.regexp_extract(
        F.lower(F.col("email")),
        r"@([^.@]+)",
        1
    )
)

In [0]:
from pyspark.sql import functions as F

store = read_csv(f"{bronze_base}/store")
store = snake_case_columns(store)

store = store.withColumn(
    "store_category",
    F.regexp_extract(
        F.lower(F.col("email_address")),
        r"@([^.@]+)",
        1
    )
)

store = normalize_date(store, "created_at")
store = normalize_date(store, "updated_at")

store_path = f"{silver_base}/store"

upsert_delta(
    store,
    store_path,
    "target.store_id = source.store_id"
)

In [0]:
display(store)

In [0]:
customer_sales = read_csv(f"{bronze_base}/sales")
customer_sales = snake_case_columns(customer_sales)

customer_sales.printSchema()
display(customer_sales)

In [0]:
customer_sales = normalize_date(customer_sales, "order_date")
customer_sales = normalize_date(customer_sales, "ship_date")

In [0]:
sales_merge_condition = """
target.order_id = source.order_id
AND target.product_id = source.product_id
"""

In [0]:
customer_sales_path = f"{silver_base}/customer_sales"

upsert_delta(
    customer_sales,
    customer_sales_path,
    sales_merge_condition
)

In [0]:
display(customer_sales)